# V10.1 vs V11 — parallel fresh40 train → isolated benchmark

**Một notebook điều phối cả hai**, không sửa model bundle. Upload notebook này và hai folder:
`MyDrive/AnchorFlow_v10_1_LiteMetric`, `MyDrive/AnchorFlow_v11_NODE`.
Data giữ ở `MyDrive/GeoLift_Data/teacher_subset_2000/`; hai cached teachers như hiện tại.
1.600 train / 400 val / 1.000 anonymous test, fresh student, max 40 + early stop.
Early stop: ít nhất 20 epoch, patience 8, minimum improvement 0.001 m.

Mặc định khởi chạy **hai tiến trình train cùng lúc**. Một GPU thì chia VRAM/compute,
không bảo đảm nhanh hơn chạy lần lượt. Hai GPU thì mỗi model một GPU.
**Final evaluate và profile luôn chạy tuần tự trên cùng GPU**, chỉ sau khi cả hai train kết thúc.
Training epoch time dưới contention không được dùng để claim isolated model speed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import gc,hashlib,importlib.util,json,os,re,shutil,subprocess,sys,time
DRIVE_DATA=Path('/content/drive/MyDrive/GeoLift_Data')
DRIVE_RUNS=Path('/content/drive/MyDrive/GeoLift_RT_Runs')
BUNDLES={
    'V10_1':Path('/content/drive/MyDrive/AnchorFlow_v10_1_LiteMetric'),
    'V11':Path('/content/drive/MyDrive/AnchorFlow_v11_NODE'),
}
LOCAL_ROOT=Path('/content/anchorflow_pair_benchmark')
WORK=LOCAL_ROOT/'shared_data' # Final per-experiment subfolder resolved with AMP/BENCH_TAG below.
BENCH_TAG='pair01' # New tag = new experiment; unchanged tag resumes both independently.
PARALLEL_TRAIN=True
TRAIN_GPU_INDICES=None # None: [0,1] if >=2 GPUs, otherwise [0,0]. Can set [0,0].
PROFILE_GPU_INDEX=0 # BOTH final evaluate/profile on this SAME GPU.
AMP_REQUEST='auto' # Common native BF16 or explicit FP32; never FP16.
BATCH_SIZE=4
ACCUMULATION=1 # On smaller GPUs, choose BATCH_SIZE=2, ACCUMULATION=2 BEFORE starting a new tag.
WORKERS=2 # Per model: parallel creates up to 2*WORKERS loader workers.
CPU_THREADS_PER_MODEL=4
SHARED_GPU_MIN_FREE_GIB=20. # Conservative preflight heuristic, NOT an OOM guarantee.
RUN_EXTRA_SELECTIONS=True # Isolated evaluate best_inverse and best_joint too.
RUN_SOLVER_AUDIT=True # Same V11 checkpoint: adaptive/default, tighter, static midpoint8.
RUN_TEST=True # Both anonymous1000 prediction ZIPs, no public GT metric.
RUN_EXPORT=False # Optional slow CPU ONNX checks; V11 export is STATIC midpoint8 only.
assert re.fullmatch(r'[A-Za-z0-9_-]+',BENCH_TAG),'BENCH_TAG: only letters, digits, underscore, hyphen'
assert AMP_REQUEST in ('auto','bf16','fp32')
assert BATCH_SIZE>=1 and ACCUMULATION>=1 and WORKERS>=0
for folder in BUNDLES.values():assert folder.is_dir(),f'Upload folder: {folder}'
for name in ('selected_2000_ids.json','kitti_trainval_2000.tar','metric_coarse_train_2000.tar',
             'relative_teacher_2000_DA3MONO_LARGE.tar'):
    assert (DRIVE_DATA/'teacher_subset_2000'/name).is_file(),f'Missing Drive data: {name}'
test_archive=DRIVE_DATA/'test_1000'/'kitti_test_1000.tar'
if not test_archive.is_file():print('Official test TAR absent: prepare will download KITTI anonymous1000, not regenerate teachers.')
assert shutil.disk_usage('/content').free/2**30>25,'Need >25GiB Colab local SSD, not PC SSD'
LOCAL_ROOT.mkdir(parents=True,exist_ok=True)

## Helper tự chứa trong notebook

Cell dưới chứa nguyên script điều phối/report; **không cần upload một folder helper thứ ba**.
Hai model chạy trong hai Python interpreter riêng để tránh trùng module `model` / `run`.
Không tự bật NVIDIA MPS, không đổi CUDA/Torch hoặc sửa source model.

In [ ]:
RUNTIME_SOURCE = '"""Independent process orchestration and matched reporting; no model imports here.\n\nCopied verbatim into the self-contained Colab notebook. Existing sealed model\nbundles are never patched. Training can overlap; final evaluation/profiling cannot.\n"""\nfrom __future__ import annotations\nimport csv\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport queue\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\n\nLABELS=(\'V10_1\',\'V11\')\nARCH_KEYS={\'architecture\',\'model_name\',\'parent_source_sha256\',\'step_size\',\'learned_step_size\',\n           \'step_min\',\'step_max\',\'step_init_logit\'}\nRUNTIME_KEYS={\'run_name\',\'drive_runs\',\'work\',\'drive_data\'}\nSHARED_SOURCES=(\'core.py\',\'support.py\',\'data.py\',\'relative_data.py\',\'relative_loss.py\',\n                \'loss_helpers.py\',\'metrics.py\',\'boundaries.py\')\n\ndef read_json(path):return json.loads(Path(path).read_text(encoding=\'utf-8\'))\ndef digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\ndef atomic_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    temporary=path.with_suffix(path.suffix+\'.partial\')\n    temporary.write_text(json.dumps(value,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    temporary.replace(path)\n\ndef verify_bundle(folder):\n    folder=Path(folder);manifest=read_json(folder/\'bundle_manifest.json\')\n    for name,expected in manifest[\'files\'].items():\n        if Path(name).name!=name or \'/\' in name or \'\\\\\' in name or name in (\'.\',\'..\'):raise ValueError(f\'Unsafe bundle filename: {name}\')\n        if name.endswith(\'.ipynb\'):continue # Mutable Colab UI only; immutable snapshot remains checked.\n        p=folder/name\n        if not p.is_file() or digest(p)!=expected:raise RuntimeError(f\'Bundle changed: {folder.name}/{name}\')\n    return manifest\n\ndef stage_bundle(folder,destination):\n    manifest=verify_bundle(folder);destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)\n    for name in [*manifest[\'files\'],\'bundle_manifest.json\']:\n        if name.endswith(\'.ipynb\'):continue\n        source,target=Path(folder)/name,destination/name\n        if source.resolve()!=target.resolve():shutil.copy2(source,target)\n    return manifest\n\ndef matched_recipe(configs):\n    a,b=(configs[label] for label in LABELS)\n    ignored=ARCH_KEYS|RUNTIME_KEYS|{k for k in set(a)|set(b) if k.startswith(\'ode_\')}\n    differences={k:{\'V10_1\':a.get(k),\'V11\':b.get(k)} for k in (set(a)|set(b))-ignored\n                 if a.get(k)!=b.get(k)}\n    if differences:raise RuntimeError(\'Unmatched training recipe: \'+json.dumps(differences,sort_keys=True))\n    for cfg in (a,b):\n        if cfg[\'init_checkpoint\'] is not None or cfg[\'epochs\']!=40:raise ValueError(\'Both experiments must be fresh student, max40\')\n        if not cfg[\'encoder_pretrained\']:raise ValueError(\'Both experiments require the same ImageNet RGB initialization policy\')\n        if cfg[\'amp\'] not in (\'bf16\',\'fp32\'):raise ValueError(\'Common native BF16 or FP32 only, no FP16\')\n        if cfg[\'flow_steps\']!=2 or not cfg[\'teacher_enabled\'] or not cfg[\'relative_enabled\']:\n            raise ValueError(\'Two-stage dual-teacher contract required\')\n        if not cfg[\'early_stopping\']:raise ValueError(\'Keep early stopping enabled for both\')\n    if a[\'model_name\']!=\'v10_lite\' or not a.get(\'learned_step_size\'):\n        raise ValueError(\'Upload the CURRENT learned-h V10.1 LiteMetric bundle, not the old fixed-h one\')\n    if b[\'model_name\']!=\'v11_node\' or b[\'ode_method\']!=\'bosh3\' or b[\'ode_terminal_time\']!=1.:\n        raise ValueError(\'V11 must use its default adaptive solver to T=1\')\n    return {\'matched_non_architecture_settings\':True,\'shared_config\':{k:a[k] for k in sorted(a) if k not in ignored},\n            \'different_state_chart_and_solver\':True,\'same_seed_does_not_force_identical_all_student_weights\':True}\n\ndef freeze_snapshot(snapshot,code,config):\n    """Preflight ALL targets before writing; never overwrite a frozen experiment."""\n    snapshot,code=Path(snapshot),Path(code)\n    manifest=verify_bundle(code)\n    content={name:(code/name).read_bytes() for name in manifest[\'files\'] if not name.endswith(\'.ipynb\')}\n    content[\'bundle_manifest.json\']=(code/\'bundle_manifest.json\').read_bytes()\n    content[\'resolved_config.json\']=json.dumps(config,indent=2,ensure_ascii=False).encode(\'utf-8\')\n    for name,value in content.items():\n        p=snapshot/name\n        if p.exists() and p.read_bytes()!=value:raise RuntimeError(f\'Frozen recipe changed: {p}; choose a new BENCH_TAG\')\n    snapshot.mkdir(parents=True,exist_ok=True)\n    for name,value in content.items():\n        p=snapshot/name\n        if not p.exists():p.write_bytes(value)\n    (code/\'resolved_config.json\').write_bytes(content[\'resolved_config.json\'])\n\ndef visible_gpu_token(index):\n    visible=os.environ.get(\'CUDA_VISIBLE_DEVICES\')\n    if visible is None:return str(index)\n    tokens=[x.strip() for x in visible.split(\',\') if x.strip()]\n    if not 0<=index<len(tokens):raise ValueError(f\'GPU index {index} outside CUDA_VISIBLE_DEVICES={visible!r}\')\n    return tokens[index]\n\ndef process_environment(gpu,code,cpu_threads=4):\n    env=os.environ.copy()\n    env.update(CUDA_VISIBLE_DEVICES=str(gpu),PYTHONUNBUFFERED=\'1\',PYTHONPATH=str(code),\n               OMP_NUM_THREADS=str(cpu_threads),MKL_NUM_THREADS=str(cpu_threads),\n               OPENBLAS_NUM_THREADS=\'1\',TOKENIZERS_PARALLELISM=\'false\')\n    return env\n\n@dataclass\nclass Command:\n    label:str\n    action:str\n    args:list\n    cwd:Path\n    local_log:Path\n    drive_log:Path\n    env:dict|None=None\n\nclass ProcessGroupError(RuntimeError):\n    def __init__(self,message,results=()):\n        super().__init__(message);self.results=list(results)\n\ndef _mirror(source,destination):\n    destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)\n    tmp=destination.with_suffix(destination.suffix+\'.partial\')\n    shutil.copy2(source,tmp);tmp.replace(destination)\n\ndef _terminate(process):\n    if process.poll() is not None:return\n    try:\n        if os.name==\'posix\':os.killpg(process.pid,signal.SIGTERM)\n        else:process.terminate()\n    except ProcessLookupError:return # Child exited between poll and signal.\n    try:process.wait(timeout=10)\n    except subprocess.TimeoutExpired:\n        try:\n            if os.name==\'posix\':os.killpg(process.pid,signal.SIGKILL)\n            else:process.kill()\n        except ProcessLookupError:pass\n        process.wait(timeout=10)\n\ndef run_processes(commands,parallel=True,status_path=None,mirror_seconds=30):\n    """Launch independent interpreters, multiplex logs, preserve a surviving peer.\n\n    A failed model does not kill the other model. Report failure AFTER peers finish.\n    User interrupt terminates only the process groups launched by this invocation.\n    Console tails mirror to Drive periodically even while a child is quiet.\n    """\n    commands=list(commands)\n    if not parallel and len(commands)>1:\n        results=[];failures=[]\n        for c in commands:\n            try:results.extend(run_processes([c],True,None,mirror_seconds))\n            except ProcessGroupError as exc:results.extend(exc.results);failures.append(str(exc))\n        if status_path:atomic_json(status_path,{\'status\':\'failed\' if failures else \'completed\',\'results\':results,\'failures\':failures})\n        if failures:raise ProcessGroupError(\'\\n\'.join(failures),results)\n        return results\n    messages=queue.Queue();active=[];results=[]\n    def reader(index,pipe):\n        try:\n            for line in iter(pipe.readline,\'\'):messages.put((index,line))\n        finally:pipe.close();messages.put((index,None))\n    def publish(status):\n        if status_path:atomic_json(status_path,{\'status\':status,\'parallel\':parallel,\n            \'active\':[{\'label\':r[\'command\'].label,\'pid\':r[\'process\'].pid} for r in active if not r[\'done\']],\n            \'results\':results})\n    try:\n        for c in commands:\n            Path(c.local_log).parent.mkdir(parents=True,exist_ok=True)\n            handle=Path(c.local_log).open(\'w\',encoding=\'utf-8\')\n            try:\n                process=subprocess.Popen(c.args,cwd=c.cwd,env=c.env,stdout=subprocess.PIPE,\n                    stderr=subprocess.STDOUT,text=True,encoding=\'utf-8\',errors=\'replace\',bufsize=1,\n                    start_new_session=os.name==\'posix\')\n            except BaseException:handle.close();raise\n            index=len(active)\n            thread=threading.Thread(target=reader,args=(index,process.stdout),daemon=True);thread.start()\n            active.append({\'command\':c,\'process\':process,\'log\':handle,\'thread\':thread,\'eof\':False,\n                           \'done\':False,\'start\':time.time(),\'last_mirror\':0.})\n            print(f\'[{c.label}/{c.action}] START pid={process.pid}\',flush=True)\n        publish(\'running\')\n        while not all(r[\'done\'] for r in active):\n            try:\n                index,line=messages.get(timeout=.25);r=active[index]\n                if line is None:r[\'eof\']=True\n                else:\n                    r[\'log\'].write(line);r[\'log\'].flush()\n                    print(f\'[{r["command"].label}/{r["command"].action}] {line}\',end=\'\',flush=True)\n            except queue.Empty:pass\n            now=time.time()\n            for r in active:\n                if r[\'done\']:continue\n                c=r[\'command\'];code=r[\'process\'].poll()\n                if now-r[\'last_mirror\']>=mirror_seconds or (code is not None and r[\'eof\']):\n                    r[\'log\'].flush();_mirror(c.local_log,c.drive_log);r[\'last_mirror\']=now\n                if code is not None and r[\'eof\']:\n                    r[\'done\']=True;r[\'log\'].close();r[\'thread\'].join(timeout=2)\n                    result={\'label\':c.label,\'action\':c.action,\'pid\':r[\'process\'].pid,\'returncode\':code,\n                            \'started_unix\':r[\'start\'],\'finished_unix\':now,\'wall_seconds\':now-r[\'start\'],\n                            \'console\':str(c.drive_log)}\n                    results.append(result);publish(\'running\')\n                    print(f\'[{c.label}/{c.action}] EXIT={code}; console={c.drive_log}\',flush=True)\n        failures=[r for r in results if r[\'returncode\']]\n        publish(\'failed\' if failures else \'completed\')\n        if failures:raise ProcessGroupError(\'One or more subprocesses failed; peers were allowed to finish:\\n\'+\n            \'\\n\'.join(f\'{r["label"]}/{r["action"]}: exit{r["returncode"]}; {r["console"]}\' for r in failures),results)\n        return results\n    except (KeyboardInterrupt,SystemExit):\n        for r in active:\n            _terminate(r[\'process\'])\n            if not r[\'log\'].closed:r[\'log\'].flush();r[\'log\'].close()\n            _mirror(r[\'command\'].local_log,r[\'command\'].drive_log)\n        publish(\'interrupted\');raise\n    except BaseException as exc:\n        # Operational orchestration failures (launch/copy), not ordinary child\n        # exit failures, must not leave our own orphan GPU workers behind.\n        for r in active:\n            if not r[\'done\']:_terminate(r[\'process\'])\n            if not r[\'log\'].closed:r[\'log\'].close()\n        if not isinstance(exc,ProcessGroupError):\n            if status_path:atomic_json(status_path,{\'status\':\'orchestration_failed\',\'error\':repr(exc),\n                \'results\':results,\'children_terminated\':True})\n        raise\n\ndef model_command(job,action,gpu=None,extra=(),suffix=\'\'):\n    label=job[\'label\'];code=Path(job[\'code\']);run_dir=Path(job[\'run_dir\'])\n    filename=f\'{action}{suffix}_console.log\'\n    return Command(label,action,[sys.executable,\'-u\',str(code/\'run.py\'),action,\'--config\',str(code/\'resolved_config.json\'),\n        \'--variant\',job[\'variant\'],*extra],code,code/filename,run_dir/filename,\n        process_environment(job[\'gpu\'] if gpu is None else gpu,code,job.get(\'cpu_threads\',4)))\n\ndef run_stage(jobs,action,parallel=False,status_path=None,gpu=None,extra=(),suffix=\'\'):\n    jobs=list(jobs);records=[]\n    # The sealed runners do not put checkpoint identity in every profiler JSON.\n    # Freeze it around the subprocess instead, so accuracy and timing cannot be\n    # accidentally joined after another run has replaced best.pth.\n    if action in (\'evaluate\',\'test\',\'profile\',\'solver_audit\',\'export\') and \'--untrained\' not in extra:\n        selection=\'rmse\'\n        if \'--checkpoint-selection\' in extra:selection=extra[extra.index(\'--checkpoint-selection\')+1]\n        for j in jobs:\n            path=Path(j[\'run_dir\'])/(\'best.pth\' if selection==\'rmse\' else f\'best_{selection}.pth\')\n            records.append((j,path,{\'action\':action,\'selection\':selection,\'checkpoint\':path.name,\n                \'checkpoint_sha256\':digest(path),\'resolved_config_sha256\':digest(Path(j[\'code\'])/\'resolved_config.json\'),\n                \'bundle_manifest_sha256\':digest(Path(j[\'code\'])/\'bundle_manifest.json\'),\n                \'GPU_visible_token\':str(j[\'gpu\'] if gpu is None else gpu)}))\n    result=run_processes([model_command(j,action,gpu,extra,suffix) for j in jobs],parallel,status_path)\n    for j,path,record in records:\n        if digest(path)!=record[\'checkpoint_sha256\']:raise RuntimeError(f\'Checkpoint changed during {action}: {path}\')\n        atomic_json(Path(j[\'run_dir\'])/f\'{action}{suffix}_artifact_record.json\',{**record,\'completed\':True})\n    return result\n\ndef verify_data_contract(contract,config):\n    if (contract[\'train\'],contract[\'val\'],contract[\'test\'])!=(1600,400,1000):raise RuntimeError(\'Split counts changed\')\n    if contract[\'manifest_sha256\']!=config[\'expected_subset_sha256\']:raise RuntimeError(\'Subset changed\')\n    if contract[\'size\']!=[352,1216] or contract[\'depth_scale\']!=256:raise RuntimeError(\'Depth/image protocol changed\')\n    if contract[\'validation_teacher_used\']:raise RuntimeError(\'Validation teacher leakage\')\n    if config[\'teacher_enabled\']:\n        r=contract[\'teacher_report\']\n        if r[\'cached_train\']!=1600 or r[\'cached_val\']!=0:raise RuntimeError(\'Metric cache must be train-only1600\')\n        if r[\'selected_teacher_content_sha256\']!=\'06bd17a84faa5020692873ee598498486d2030f588b7429321fa0a70d9ebb7a6\':\n            raise RuntimeError(\'Metric teacher content changed\')\n    if config[\'relative_enabled\']:\n        r=contract[\'relative_teacher\']\n        if r[\'cached_train\']!=1600 or r[\'cached_val\']!=0 or contract[\'relative_val_used\']:raise RuntimeError(\'Relative cache leakage\')\n        if r[\'selected_content_sha256\']!=\'dd3b49a4474560fedbfc78504dd584c0234d790c65a356c2e74b6bb82759e86a\':\n            raise RuntimeError(\'Relative teacher content changed\')\n\ndef gpu_snapshot():\n    try:\n        p=subprocess.run([\'nvidia-smi\'],capture_output=True,text=True,timeout=10)\n        return {\'returncode\':p.returncode,\'stdout\':p.stdout,\'stderr\':p.stderr}\n    except (OSError,subprocess.TimeoutExpired) as e:return {\'unavailable\':str(e)}\n\ndef _csv_rows(path):\n    with Path(path).open(encoding=\'utf-8\',newline=\'\') as f:return list(csv.DictReader(f))\n\ndef _accuracy(label,selection,report):\n    score=report[\'final\'];a=score[\'all\']\n    return {\'model\':label,\'selection\':selection,\'epoch\':report.get(\'checkpoint_epoch\',report.get(\'epoch\')),\n        \'RMSE_m\':a[\'rmse_m\'],\'MAE_m\':a[\'mae_m\'],\'iRMSE_km_inv\':a[\'irmse_km_inv\'],\n        \'iMAE_km_inv\':a[\'imae_km_inv\'],\'AbsRel\':a[\'abs_rel\'],\'delta1\':a[\'delta1\'],\'pixels\':a[\'pixels\'],\n        **{f\'RMSE_{k}_m\':score[k][\'rmse_m\'] for k in (\'0-20\',\'20-40\',\'40-60\',\'60-80\',\'80-120\',\'edge\')},\n        \'GT_boundary3_m\':report[\'gt_boundary\'][\'bands\'][\'3\'][\'rmse_m\'],\n        \'joint_target_score\':max(a[\'rmse_m\']/.9,a[\'irmse_km_inv\']/3.2),\n        \'both_targets\':a[\'rmse_m\']<.9 and a[\'irmse_km_inv\']<3.2}\n\ndef _write_csv(path,rows):\n    if not rows:return\n    names=list(dict.fromkeys(k for r in rows for k in r))\n    with Path(path).open(\'w\',encoding=\'utf-8\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=names);writer.writeheader();writer.writerows(rows)\n\ndef compare_reports(jobs,root,expected_pixels=25424992):\n    """Accuracy from isolated final reevaluation, runtime from isolated real100.\n\n    Both model checkpoint selections stay explicit; never combine metrics from\n    different checkpoints. Parallel training epoch duration is NOT isolated speed.\n    """\n    jobs=list(jobs)\n    if tuple(j[\'label\'] for j in jobs)!=LABELS:raise ValueError(\'Expected ordered V10_1/V11 pair\')\n    matched_recipe({j[\'label\']:j[\'config\'] for j in jobs})\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    primary=[];selections=[];efficiency=[];budget=[];main_profiles=[];curves={};solver_rows=[]\n    stages=[];tails=[];boundary_rows=[];policies=[];provenance=[]\n    for job in jobs:\n        label=job[\'label\'];path=Path(job[\'run_dir\']);cfg=job[\'config\']\n        report=read_json(path/\'val_metrics.json\')\n        if report[\'samples\']!=400 or report[\'teacher_at_inference\'] or report[\'checkpoint_selection\']!=\'rmse\':\n            raise RuntimeError(f\'{label}: invalid evaluation protocol\')\n        evaluation=read_json(path/\'evaluate_artifact_record.json\');profile_record=read_json(path/\'profile_artifact_record.json\')\n        for key in (\'checkpoint_sha256\',\'resolved_config_sha256\',\'bundle_manifest_sha256\',\'GPU_visible_token\'):\n            if evaluation[key]!=profile_record[key]:raise RuntimeError(f\'{label}: evaluation/profile provenance mismatch: {key}\')\n        if not evaluation[\'completed\'] or not profile_record[\'completed\']:raise RuntimeError(\'Incomplete artifact stage\')\n        if digest(path/\'best.pth\')!=profile_record[\'checkpoint_sha256\']:raise RuntimeError(\'Best checkpoint changed since profiling\')\n        provenance.append({\'model\':label,\'checkpoint_epoch\':report[\'checkpoint_epoch\'],**profile_record})\n        if report[\'final\'][\'all\'][\'pixels\']!=expected_pixels:raise RuntimeError(f\'{label}: GT support changed\')\n        primary.append(_accuracy(label,\'rmse\',report))\n        for selection in (\'rmse\',\'inverse\',\'joint\'):\n            file=path/(\'val_metrics.json\' if selection==\'rmse\' else f\'val_metrics_{selection}.json\')\n            if file.is_file():\n                r=read_json(file)\n                if r[\'final\'][\'all\'][\'pixels\']!=expected_pixels:raise RuntimeError(\'Selection GT support changed\')\n                evidence=read_json(path/(\'evaluate_artifact_record.json\' if selection==\'rmse\' else f\'evaluate_{selection}_artifact_record.json\'))\n                checkpoint=path/(\'best.pth\' if selection==\'rmse\' else f\'best_{selection}.pth\')\n                if digest(checkpoint)!=evidence[\'checkpoint_sha256\']:raise RuntimeError(\'Selection checkpoint changed since evaluation\')\n                if evidence[\'resolved_config_sha256\']!=evaluation[\'resolved_config_sha256\']:\n                    raise RuntimeError(\'Selection config changed\')\n                selections.append(_accuracy(label,selection,r))\n        rows=_csv_rows(path/\'train_log.csv\')\n        if not rows:raise RuntimeError(f\'No trained epochs for {label}\')\n        epochs=[int(r[\'epoch\']) for r in rows]\n        if epochs!=list(range(max(epochs)+1)):raise RuntimeError(f\'{label}: duplicate/missing epoch logs\')\n        curves[label]=rows\n        status=read_json(path/\'training_status.json\')\n        if status[\'status\'] not in (\'max_epochs_completed\',\'early_stopped\') or status[\'epochs_completed\']!=len(rows):\n            raise RuntimeError(f\'{label}: training status incomplete; rerun train cell to resume, then reevaluate\')\n        budget.append({\'model\':label,\'epochs_logged\':len(rows),\'last_epoch_logged\':max(epochs),\n            \'optimizer_updates\':int(rows[-1][\'global_step\']),\'batch_size\':cfg[\'batch_size\'],\n            \'accumulation\':cfg[\'accumulation\'],\'effective_batch\':cfg[\'batch_size\']*cfg[\'accumulation\'],\n            \'epoch_seconds_mean_UNDER_TRAIN_CONTENTION\':sum(float(r[\'epoch_seconds\']) for r in rows)/len(rows),\n            \'train_seconds_total_UNDER_TRAIN_CONTENTION\':sum(float(r[\'train_seconds\']) for r in rows),\n            \'training_status\':status[\'status\']})\n        p=read_json(path/\'profile.json\');main_profiles.append(p)\n        real=p[\'real_scene_profile\']\n        if real.get(\'samples\')!=100 or real.get(\'indices\')!=\'i*13 mod400\':raise RuntimeError(\'Unmatched real-scene profile\')\n        if p[\'untrained_weights\'] or p[\'batch\']!=1 or p[\'shape\']!=[352,1216]:raise RuntimeError(\'Invalid inference profile\')\n        nfe=real.get(\'solver\',{}).get(\'nfe_mean\',real.get(\'executed_nfe\'))\n        efficiency.append({\'model\':label,\'graph\':\'primary adaptive RK3(2)\' if label==\'V11\' else \'primary learned-h Euler2\',\n            \'parameters\':p[\'total_parameters\'],\'synthetic_conv_linear_GMAC\':p[\'total_conv_linear_macs\']/1e9,\n            \'real100_wall_median_ms\':real[\'wall_median_ms\'],\'real100_wall_P95_ms\':real[\'wall_p95_ms\'],\n            \'real100_actual_NFE_mean\':nfe,\'peak_torch_allocated_MiB\':p[\'peak_cuda_allocated_mib\'],\n            \'GPU\':p[\'device\'],\'CNN_precision\':p[\'precision\'],\'torch\':p[\'torch\'],\n            \'RMSE_m\':report[\'final\'][\'all\'][\'rmse_m\'],\'iRMSE_km_inv\':report[\'final\'][\'all\'][\'irmse_km_inv\'],\n            \'latency_measured_without_peer_training\':True})\n        if label==\'V11\':\n            solver=path/\'solver_comparison.json\'\n            if solver.is_file():solver_rows=read_json(solver)\n            fixed_p,fixed_r=path/\'profile_fixed_midpoint8.json\',path/\'val_metrics_fixed_midpoint8.json\'\n            if fixed_p.is_file() and fixed_r.is_file():\n                pp,rr=read_json(fixed_p),read_json(fixed_r);real=pp[\'real_scene_profile\']\n                if rr[\'checkpoint_epoch\']!=report[\'checkpoint_epoch\']:raise RuntimeError(\'Static candidate used a different checkpoint\')\n                audit=read_json(path/\'solver_audit_artifact_record.json\')\n                if audit[\'checkpoint_sha256\']!=profile_record[\'checkpoint_sha256\']:raise RuntimeError(\'Static solver audit used different weights\')\n                if rr[\'final\'][\'all\'][\'pixels\']!=expected_pixels:raise RuntimeError(\'Static solver GT support changed\')\n                for key in (\'device\',\'torch\',\'precision\',\'channels_last\',\'runs\',\'shape\',\'batch\',\'compiled_total\'):\n                    if pp[key]!=p[key]:raise RuntimeError(f\'Static profile protocol mismatch: {key}\')\n                if real[\'samples\']!=100 or real[\'indices\']!=\'i*13 mod400\':raise RuntimeError(\'Static real-scene profile changed\')\n                efficiency.append({\'model\':\'V11\',\'graph\':\'static midpoint8 (different numerical solver)\',\n                    \'parameters\':pp[\'total_parameters\'],\'synthetic_conv_linear_GMAC\':pp[\'total_conv_linear_macs\']/1e9,\n                    \'real100_wall_median_ms\':real[\'wall_median_ms\'],\'real100_wall_P95_ms\':real[\'wall_p95_ms\'],\n                    \'real100_actual_NFE_mean\':real[\'solver\'][\'nfe_mean\'],\'peak_torch_allocated_MiB\':pp[\'peak_cuda_allocated_mib\'],\n                    \'GPU\':pp[\'device\'],\'CNN_precision\':pp[\'precision\'],\'torch\':pp[\'torch\'],\n                    \'RMSE_m\':rr[\'final\'][\'all\'][\'rmse_m\'],\'iRMSE_km_inv\':rr[\'final\'][\'all\'][\'irmse_km_inv\'],\n                    \'latency_measured_without_peer_training\':True})\n        stages.extend({\'model\':label,\'stage\':name,**score} for name,score in report[\'stage_native_gt_metrics\'].items())\n        tails.extend({\'model\':label,\'absolute_error_threshold_m\':threshold,**score}\n                     for threshold,score in report[\'final\'][\'all\'][\'error_tail\'].items())\n        boundary_rows.extend({\'model\':label,\'GT_boundary_radius_px\':radius,**{k:v for k,v in score.items() if k!=\'bad_pixel_rates\'}}\n                             for radius,score in report[\'gt_boundary\'][\'bands\'].items())\n        policies.extend({\'model\':label,\'policy\':name,**{k:v for k,v in report[name][\'all\'].items() if k!=\'error_tail\'}}\n                        for name in (\'final\',\'pre_anchor\',\'legacy_hard_anchor\'))\n    for key in (\'device\',\'torch\',\'precision\',\'channels_last\',\'runs\',\'shape\',\'batch\',\'compiled_total\'):\n        if main_profiles[0][key]!=main_profiles[1][key]:raise RuntimeError(f\'Profile hardware/protocol mismatch: {key}\')\n    for key in (\'cudnn_enabled\',\'cudnn_benchmark\',\'cudnn_deterministic\',\'cudnn_allow_tf32\',\'matmul_allow_tf32\',\'cuda_build\',\'cudnn_version\'):\n        if main_profiles[0][\'runtime_backend\'][key]!=main_profiles[1][\'runtime_backend\'][key]:\n            raise RuntimeError(f\'Profile backend mismatch: {key}\')\n    common_epochs=min(len(r) for r in curves.values());common=[]\n    for label,rows in curves.items():\n        r=min(rows[:common_epochs],key=lambda r:float(r[\'val_rmse_m\']))\n        common.append({\'model\':label,\'common_completed_epochs\':common_epochs,\'selected_epoch\':int(r[\'epoch\']),\n                       \'RMSE_m\':float(r[\'val_rmse_m\']),\'iRMSE_km_inv_SAME_EPOCH\':float(r[\'val_irmse_km_inv\']),\n                       \'source\':\'epoch validation log, not isolated reevaluation; checkpoint may not have been retained\'})\n    a,b=primary\n    delta={\'V11_minus_V10_1_RMSE_m\':b[\'RMSE_m\']-a[\'RMSE_m\'],\n           \'V11_minus_V10_1_iRMSE_km_inv\':b[\'iRMSE_km_inv\']-a[\'iRMSE_km_inv\'],\n           \'RMSE_improvement_percent\':100*(a[\'RMSE_m\']-b[\'RMSE_m\'])/a[\'RMSE_m\'],\n           \'metric_SSE_reduction_percent\':100*(1-(b[\'RMSE_m\']/a[\'RMSE_m\'])**2),\n           \'real100_latency_V11_over_V10_1\':efficiency[1][\'real100_wall_median_ms\']/efficiency[0][\'real100_wall_median_ms\']}\n    for name,rows in [(\'comparison_accuracy.csv\',primary),(\'comparison_checkpoint_selections.csv\',selections),\n                      (\'comparison_efficiency.csv\',efficiency),(\'comparison_training_budget.csv\',budget),\n                      (\'comparison_common_epoch_budget.csv\',common),(\'comparison_v11_solvers.csv\',solver_rows),\n                      (\'comparison_stages.csv\',stages),(\'comparison_tails.csv\',tails),\n                      (\'comparison_GT_boundaries.csv\',boundary_rows),(\'comparison_sensor_policies.csv\',policies)]:\n        _write_csv(root/name,rows)\n    atomic_json(root/\'comparison_summary.json\',{\'primary\':primary,\'selections\':selections,\'efficiency\':efficiency,\n        \'training_budget\':budget,\'common_epoch_budget\':common,\'deltas\':delta,\n        \'artifact_provenance\':provenance,\n        \'interpretation\':\'Same data/loss/hyperparameter policy, single seed. Different chart/solver, student initialization order and stopping epochs; not a pure solver ablation. Parallel epoch duration is not isolated model speed.\'})\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig,axes=plt.subplots(2,2,figsize=(13,9))\n    for label,rows in curves.items():\n        x=[int(r[\'epoch\'])+1 for r in rows]\n        for ax,key,title in [(axes[0,0],\'val_rmse_m\',\'Global validation RMSE (m)\'),\n                              (axes[0,1],\'val_irmse_km_inv\',\'Validation iRMSE (km^-1)\'),\n                              (axes[1,0],\'val_native_D4_rmse_m\',\'Native quarter-grid RMSE (m)\')]:\n            ax.plot(x,[float(r[key]) for r in rows],label=label);ax.set_title(title)\n    axes[0,0].axhline(.9,color=\'red\',ls=\'--\',alpha=.5);axes[0,1].axhline(3.2,color=\'red\',ls=\'--\',alpha=.5)\n    for r in efficiency:\n        axes[1,1].scatter(r[\'real100_wall_median_ms\'],r[\'RMSE_m\'],s=70)\n        axes[1,1].annotate(r[\'model\']+\' \'+(\'static8\' if \'static\' in r[\'graph\'] else \'main\'),\n                         (r[\'real100_wall_median_ms\'],r[\'RMSE_m\']),xytext=(4,6),textcoords=\'offset points\')\n    axes[1,1].set_xlabel(\'Isolated real100 wall median (ms)\');axes[1,1].set_ylabel(\'RMSE (m)\')\n    axes[1,1].set_title(\'Same-checkpoint accuracy / latency\')\n    for ax in axes.flat:ax.grid(alpha=.25)\n    for ax in (axes[0,0],axes[0,1],axes[1,0]):ax.legend();ax.set_xlabel(\'Completed epochs\')\n    fig.tight_layout();fig.savefig(root/\'comparison_dashboard.png\',dpi=160);plt.close(fig)\n    lines=[\'# V10.1 / V11 benchmark\',\'\',\n        \'| Model | Best RMSE m | iRMSE same checkpoint | Best epoch |\',\n        \'|---|---:|---:|---:|\',\n        *[f\'| {r["model"]} | {r["RMSE_m"]:.6f} | {r["iRMSE_km_inv"]:.6f} | {r["epoch"]} |\' for r in primary],\n        \'\',f\'V11 RMSE improvement: {delta["RMSE_improvement_percent"]:.2f}%; negative means worse.\',\n        f\'V11/main real100 wall latency ratio: {delta["real100_latency_V11_over_V10_1"]:.3f}x.\',\n        \'\',f\'Common logged budget: {common_epochs}epochs; see comparison_common_epoch_budget.csv.\',\n        \'\', \'Training was scheduled in independent subprocesses; epoch durations under overlap are not isolated model speed.\',\n        \'Final evaluation/profile ran sequentially on the same GPU. Anonymous1000 has no public GT.\',\n        \'V11 static8 is a different numerical solver, evaluated separately. Single seed, not pure solver-only causal evidence.\']\n    (root/\'COMPARISON.md\').write_text(\'\\n\'.join(lines)+\'\\n\',encoding=\'utf-8\')\n    return read_json(root/\'comparison_summary.json\')\n'
RUNTIME_SHA256 = '13488682d13cb03858f7662571ac4f6e1621d2d618f44a127ab55c65259fe59f'

runner_file=LOCAL_ROOT/'anchorflow_pair_benchmark.py'
assert hashlib.sha256(RUNTIME_SOURCE.encode('utf-8')).hexdigest()==RUNTIME_SHA256
runner_file.write_text(RUNTIME_SOURCE,encoding='utf-8')
spec=importlib.util.spec_from_file_location('anchorflow_pair_benchmark',runner_file)
bench=importlib.util.module_from_spec(spec);sys.modules[spec.name]=bench;spec.loader.exec_module(bench)
print('Coordinator ready:',runner_file)


## Verify both bundles → install common requirements → CPU contracts

Checksum bỏ qua notebook UI `.ipynb` (Colab thay metadata/output), nhưng vẫn strict với mọi
code/config/notebook snapshot. Không bypass test failure. Cài cùng environment, không reinstall Torch.

In [ ]:
manifests={};CODES={}
for label,folder in BUNDLES.items():
    CODES[label]=LOCAL_ROOT/('code_'+label)
    manifests[label]=bench.stage_bundle(folder,CODES[label])
for name in bench.SHARED_SOURCES:
    assert (CODES['V10_1']/name).read_bytes()==(CODES['V11']/name).read_bytes(),f'Shared primitive differs: {name}'
args=[sys.executable,'-m','pip','install','-q']
for folder in CODES.values():args.extend(['-r',str(folder/'requirements.txt')])
subprocess.run(args,check=True)
import torch,torchdiffeq
assert torch.cuda.is_available(),'Select GPU runtime in Colab'
assert torchdiffeq.__version__=='0.2.5'
for label,folder in CODES.items():
    tested=subprocess.run([sys.executable,'-u','-m','unittest','discover','-s',str(folder),'-v'],cwd=folder,
        capture_output=True,text=True,encoding='utf-8',errors='replace',
        env=bench.process_environment(bench.visible_gpu_token(PROFILE_GPU_INDEX),folder,CPU_THREADS_PER_MODEL))
    print(label,tested.stdout+tested.stderr)
    assert tested.returncode==0,f'{label} tests failed; traceback above'

## Match recipe and freeze source/config

Same seed42, max40, pretrained RGB policy, batch/accumulation, optimizer/loss/teachers/early stop.
V10.1 giữ learned h + fixed2 Euler; V11 giữ adaptive RK3(2) + smooth chart + fixedT1.
Đây là architecture bundle comparison, **không phải pure solver-only ablation** hoặc identical-all-weights init.
Không đổi batch/loss/precision/source giữa chừng để resume. PARALLEL_TRAIN có thể đổi sang False
để tránh contention khi resume; scheduling history được log riêng.

In [ ]:
n_gpu=torch.cuda.device_count()
train_indices=TRAIN_GPU_INDICES if TRAIN_GPU_INDICES is not None else ([0,1] if n_gpu>=2 else [0,0])
assert len(train_indices)==2 and all(isinstance(i,int) and 0<=i<n_gpu for i in train_indices)
assert 0<=PROFILE_GPU_INDEX<n_gpu
involved=sorted(set(train_indices+[PROFILE_GPU_INDEX]))
gpu_info=[]
for i in involved:
    with torch.cuda.device(i):
        free,total=torch.cuda.mem_get_info()
        native=torch.cuda.get_device_capability(i)[0]>=8 and torch.cuda.is_bf16_supported(including_emulation=False)
    gpu_info.append({'logical_index':i,'token':bench.visible_gpu_token(i),'name':torch.cuda.get_device_name(i),
                     'native_bf16':native,'free_GiB':free/2**30,'total_GiB':total/2**30})
common_native=all(r['native_bf16'] for r in gpu_info)
if AMP_REQUEST=='bf16':assert common_native,'Select fp32; every training/profile GPU must support native BF16'
ACTUAL_AMP=('bf16' if common_native else 'fp32') if AMP_REQUEST=='auto' else AMP_REQUEST
shared_gpu=PARALLEL_TRAIN and train_indices[0]==train_indices[1]
if shared_gpu:
    info=next(r for r in gpu_info if r['logical_index']==train_indices[0])
    assert info['free_GiB']>=SHARED_GPU_MIN_FREE_GIB,(
        f"Shared GPU free={info['free_GiB']:.1f}GiB < conservative guard={SHARED_GPU_MIN_FREE_GIB}. "
        'Use PARALLEL_TRAIN=False; or before starting a NEW tag choose batch2/accum2 and a justified lower guard. '
        'No automatic batch/precision fallback.')
    print('WARNING: both train jobs share one GPU. More VRAM and possible time-slicing; NOT guaranteed faster.')
PROFILE_GPU=bench.visible_gpu_token(PROFILE_GPU_INDEX)
COMPARE_ROOT=DRIVE_RUNS/f'Compare_V10_1_V11_Fresh40_{ACTUAL_AMP}_{BENCH_TAG}'
# Isolate local checkpoints/logs too: runner stores them under WORK/runs/<model>.
# A NEW tag must never append epoch0 to a prior experiment's local train CSV.
WORK=LOCAL_ROOT/'shared_data'/f'{ACTUAL_AMP}_{BENCH_TAG}'
configs={};JOBS=[]
for label,index in zip(bench.LABELS,train_indices):
    cfg=bench.read_json(CODES[label]/'config.json')
    cfg.update(drive_data=str(DRIVE_DATA),drive_runs=str(COMPARE_ROOT),work=str(WORK),run_name=label,
        epochs=40,seed=42,batch_size=BATCH_SIZE,accumulation=ACCUMULATION,workers=WORKERS,
        amp=ACTUAL_AMP,amp_request=AMP_REQUEST,init_checkpoint=None,encoder_pretrained=True,
        teacher_enabled=True,relative_enabled=True,checkpoint_selection='rmse',compile=False,
        profile_warmup=30,profile_runs=100)
    configs[label]=cfg
recipe=bench.matched_recipe(configs)
for label,index in zip(bench.LABELS,train_indices):
    cfg=configs[label]
    bench.freeze_snapshot(COMPARE_ROOT/'source_bundle'/label,CODES[label],cfg)
    JOBS.append({'label':label,'code':str(CODES[label]),'variant':'dual_teacher','config':cfg,
                 'run_dir':str(COMPARE_ROOT/label/'dual_teacher'),'gpu':bench.visible_gpu_token(index),
                 'cpu_threads':CPU_THREADS_PER_MODEL})
frozen_runner=COMPARE_ROOT/'source_bundle'/'anchorflow_pair_benchmark.py'
if frozen_runner.exists():assert frozen_runner.read_bytes()==runner_file.read_bytes(),'Coordinator changed; choose a new BENCH_TAG'
else:shutil.copy2(runner_file,frozen_runner)
execution={'utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'parallel_train':PARALLEL_TRAIN,
           'train_gpu_indices':train_indices,'profile_gpu_index':PROFILE_GPU_INDEX,'gpu_info':gpu_info,
           'cpu_threads_per_model':CPU_THREADS_PER_MODEL,'shared_gpu':shared_gpu,
           'torch':str(torch.__version__),'coordinator_sha256':RUNTIME_SHA256,'recipe':recipe}
bench.atomic_json(COMPARE_ROOT/f'execution_{time.time_ns()}.json',execution)
bench.atomic_json(COMPARE_ROOT/'benchmark_recipe.json',recipe)
print(json.dumps(execution,indent=2));print('COMPARISON OUTPUT:',COMPARE_ROOT)

## Prepare shared data ONCE, smoke both SEQUENTIALLY

Loader/cache sources của hai bundle được kiểm tra byte-identical. Prepare/audit dùng V10.1 một lần;
cả hai sau đó chỉ đọc chung RGB/sparse/GT/K và train-only teacher caches.
Không run prepare/smoke đồng thời. Không generate lại relative teacher.

In [ ]:
bench.run_stage([JOBS[0]],'prepare',gpu=PROFILE_GPU)
contract=bench.read_json(WORK/'data_contract.json')
for cfg in configs.values():bench.verify_data_contract(contract,cfg)
bench.atomic_json(COMPARE_ROOT/'data_contract.json',contract)
for job in JOBS:
    since=time.time()
    try:
        bench.run_stage([job],'smoke',gpu=PROFILE_GPU)
    finally:
        diagnostic=WORK/'gpu_failure_report.json'
        if diagnostic.exists() and diagnostic.stat().st_mtime>=since:
            shutil.copy2(diagnostic,Path(job['run_dir'])/'smoke_gpu_failure_report.json')
    report=bench.read_json(WORK/'smoke_report.json')
    bench.atomic_json(Path(job['run_dir'])/'smoke_report.json',report)
    print(job['label'],'SMOKE PASS:',report)
gc.collect();torch.cuda.empty_cache()

## RUN BOTH TRAIN JOBS — một cell

Console có prefix [V10_1/train] / [V11/train], tự cập nhật Drive console mỗi30giây.
Checkpoint/CSV/JSONL sync mỗi epoch vào folder model riêng. Một model lỗi: model kia vẫn chạy,
cell báo thất bại sau khi peer kết thúc; không bỏ NaN, không tự đổi precision/batch.
Stop cell sẽ dừng hai nhóm tiến trình do cell tạo; checkpoint hoàn chỉnh gần nhất được giữ.
Colab ngắt: chạy lại notebook cùng BENCH_TAG/config, hai model tự resume độc lập.
Không mở hai notebook cùng train vào cùng COMPARE_ROOT. Nếu OOM: chờ peer kết thúc,
chọn PARALLEL_TRAIN=False và chạy lại; giữ nguyên batch/precision/config của experiment.

In [ ]:
bench.run_stage(JOBS,'train',parallel=PARALLEL_TRAIN,status_path=COMPARE_ROOT/'train_pair_status.json')
for job in JOBS:
    status=bench.read_json(Path(job['run_dir'])/'training_status.json')
    print(job['label'],status)
print('BOTH TRAIN PROCESSES EXITED. Final evaluation/profile can now run without peer contention.')
bench.atomic_json(COMPARE_ROOT/'gpu_after_training.json',bench.gpu_snapshot())

## Final evaluate / test / profile — KHÔNG SONG SONG

Cả hai evaluate/profile trên cùng PROFILE_GPU, cùng precision/B1/image-size/real100 scenes.
BestRMSE, bestInverse và bestJoint luôn giữ metric của đúng checkpoint đó.
V11 static-midpoint8 được đo như một solver approximation khác, không trộn với adaptive accuracy.
Anonymous1000 test không có public GT; không gọi validation metric là leaderboard result.

In [ ]:
bench.run_stage(JOBS,'evaluate',parallel=False,gpu=PROFILE_GPU,
                status_path=COMPARE_ROOT/'evaluate_pair_status.json')
if RUN_EXTRA_SELECTIONS:
    for selection in ('inverse','joint'):
        bench.run_stage(JOBS,'evaluate',parallel=False,gpu=PROFILE_GPU,
                        extra=('--checkpoint-selection',selection),suffix='_'+selection)
if RUN_SOLVER_AUDIT:bench.run_stage([JOBS[1]],'solver_audit',gpu=PROFILE_GPU)
if RUN_TEST:bench.run_stage(JOBS,'test',parallel=False,gpu=PROFILE_GPU,
                          status_path=COMPARE_ROOT/'test_pair_status.json')
bench.atomic_json(COMPARE_ROOT/'gpu_before_isolated_profiles.json',bench.gpu_snapshot())
bench.run_stage(JOBS,'profile',parallel=False,gpu=PROFILE_GPU,
                status_path=COMPARE_ROOT/'profile_pair_status.json')
if RUN_EXPORT:bench.run_stage(JOBS,'export',parallel=False,gpu=PROFILE_GPU)

## Compare accuracy, real latency, budget, same-checkpoint solver trade-off

Inference runtime dùng real100 **wall median/P95**, không dùng training epoch_seconds đang tranh GPU.
Early stop có thể khác số epoch: báo actual budget và best trong common logged epoch prefix riêng.
Common-prefix là epoch log, không giả vờ checkpoint của epoch đó còn tồn tại.

In [ ]:
summary=bench.compare_reports(JOBS,COMPARE_ROOT)
import pandas as pd
from IPython.display import display,Image,Markdown
for file in ('comparison_accuracy.csv','comparison_checkpoint_selections.csv','comparison_efficiency.csv',
             'comparison_training_budget.csv','comparison_common_epoch_budget.csv','comparison_stages.csv',
             'comparison_tails.csv','comparison_GT_boundaries.csv','comparison_sensor_policies.csv'):
    print(file);display(pd.read_csv(COMPARE_ROOT/file))
solver=COMPARE_ROOT/'comparison_v11_solvers.csv'
if solver.exists():display(pd.read_csv(solver))
print('DELTA:',json.dumps(summary['deltas'],indent=2))
display(Image(filename=str(COMPARE_ROOT/'comparison_dashboard.png')))
display(Markdown((COMPARE_ROOT/'COMPARISON.md').read_text(encoding='utf-8')))
print('REPORTS:',COMPARE_ROOT)
for job in JOBS:print(job['label'],'logs/checkpoints/test:',job['run_dir'])